# Final Evaluation

This notebook contains the comprehensive benchmarking and evaluation of the proposed framework using GPT-2 and Llama-3-8B. The evaluation is structured into Tiers as defined in the research:

*   **Tier 1:** Per-Persona Reward Accuracy (Static Evaluation).
*   **Tier 2:** Z-Anchor Drift Tracking (Dynamic Adaptation Evaluation).
*   **Ablation Study:** Impact of MMP (Multi-Modal Persona) and FiLm layers.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/vpl_llm_2/
!ls -F

/content/drive/MyDrive/vpl_llm_2
args/
baseline_bash.sh
convert_user_data.py
create_multi_persona_simulation.py
custom_vpl_generation.ipynb
data/
evaluate_adaptation_velocity.py
evaluate_reward_scores.py
FYP_VPL_V1.ipynb
generate_llm_embeddings_grouped_personas.sh
generate_llm_embeddings_personas_ab.sh
GPT_2_emebeding_creation_and_Benchmarking.ipynb
hidden_context/
llama_embeding_creation.ipynb
logs/
__pycache__/
results/
run_all_drift_evals.py
run_generation.py
sigma_sanity_check.py
Simulation.ipynb
simulation_plot.png
simulation_plot_reward_fluctuation.png
simulation_reward_fluctuation.png
submit_job_grouped_personas.sh
submit_job_personas_ab.sh
temp_gpt2_embed_file.sh
temp_gpt2_run.sh
temp_llama_run.sh
vae_session_inference.py
vpl_args_eval.png
wandb/


## 1. GPT-2 Benchmarking

This section evaluates the proposed model's performance using GPT-2 as the base model. We compare our proposed apporach (with FiLm and MMP) against several baselines:
*   **Static VPL:** VPL without MMP or FiLm.
*   **BTL Baseline:** Bradley-Terry-Luce model.
*   **DPL Baseline:** Fixed categorical/mean and variance persona embeddings.

### Analysis of GPT-2 Tier 1 Results

Based on the execution outputs:
*   **Proposed Model (with MMP/FiLm):** Achieved a mean accuracy of **70.7%**, showing significant robustness across all personas (A-E).
*   **Static VPL:** Achieved only **56.8%**, with notable failures in Persona B and E, highlighting the importance of dynamic context.
*   **Baselines:** The BTL (51.4%) and Categorical (52.2%) baselines perform near chance on specific personas, validating the VPL approach.

"logs/FYP_final/grouped_personas_2_gpt_2_vpl_without_MMP_without_Film"

In [3]:
!python /content/drive/MyDrive/vpl_llm_2/evaluate_reward_scores.py \
--vae_model_path  /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/grouped_personas_2_gpt_2_vpl_without_MMP_without_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt \
--sim_dir         /content/drive/MyDrive/evaluation/Final_balanced_personas_gpt_2_without_MMP \
--n_context       8 \
--output_csv      /content/drive/MyDrive/evaluation/results/final_gpt2_without_MMP_without_film_eval.csv

[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading VAE model from /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/grouped_personas_2_gpt_2_vpl_without_MMP_without_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1697      67.9%       0.1254      0.3117
Persona B     2500      909      36.4%      -0.0506      0.3257
Persona C    

logs/gpt2_Persona_2_survey_100_film_MMP

In [4]:
!python /content/drive/MyDrive/vpl_llm_2/evaluate_reward_scores.py \
--vae_model_path  /content/drive/MyDrive/vpl_llm_2/logs/gpt2_Persona_2_survey_100_film_MMP/all/vae_gpt2__0_0.0001_cosine_5_0.001_512_768_seed0_peft_last_checkpoint/model.pt \
--sim_dir         /content/drive/MyDrive/evaluation/Final_balanced_personas_gpt2_MMP \
--n_context       8 \
--output_csv      /content/drive/MyDrive/evaluation/results/final_gpt2_with_Film_with_MMP_eval.csv

[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading VAE model from /content/drive/MyDrive/vpl_llm_2/logs/gpt2_Persona_2_survey_100_film_MMP/all/vae_gpt2__0_0.0001_cosine_5_0.001_512_768_seed0_peft_last_checkpoint/model.pt …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1667      66.7%       0.3879      1.0281
Persona B     2500     1616      64.6%       0.3835      0.9620
Persona C     2500     1775      71.0%    

log_dir=logs/grouped_personas__gpt_2_base_baseline_Film

In [5]:
%cd /content/drive/MyDrive/vpl_llm_2

# Using --baseline_embed_dim 768 to match the GPT-2 checkpoint dimensions per script usage
!python evaluate_reward_scores.py \
    --model_type base \
    --baseline_embed_dim 768 \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_base_baseline_Film/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint" \
    --sim_dir "/content/drive/MyDrive/evaluation/Final_balanced_personas_gpt_2_without_MMP" \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_gpt2_baseline_BTL_tier1_eval.csv"

/content/drive/MyDrive/vpl_llm_2
[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading Baseline base model from /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_base_baseline_Film/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1487      59.5%       0.1005      0.6281
Persona B     2500      950      38.0%      -0.1051      0.6050
Persona C     2500

logs/grouped_personas__gpt_2_categorical_baseline_Film

In [6]:
%cd /content/drive/MyDrive/vpl_llm_2

# Changing --model_type to categorical to match the checkpoint architecture
!python evaluate_reward_scores.py \
    --model_type categorical \
    --baseline_embed_dim 768 \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_categorical_baseline_Film/all/categorical_gpt2__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint" \
    --sim_dir "/content/drive/MyDrive/evaluation/Final_balanced_personas_gpt_2_without_MMP" \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_gpt2_baseline_categorical_tier1_eval.csv"

/content/drive/MyDrive/vpl_llm_2
[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading Baseline categorical model from /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_categorical_baseline_Film/all/categorical_gpt2__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1506      60.2%       0.0137      0.0522
Persona B     2500      867      34.7%      -0.0133   

logs/grouped_personas__gpt_2_mean_and_var_baseline_Film

In [7]:
%cd /content/drive/MyDrive/vpl_llm_2

# Changing --model_type to categorical to match the checkpoint architecture
!python evaluate_reward_scores.py \
    --model_type mean_and_variance \
    --baseline_embed_dim 768 \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_mean_and_var_baseline_Film/all/mean_and_variance_gpt2__0_0.0001_cosine_2_0.0_seed0_last_checkpoint" \
    --sim_dir "/content/drive/MyDrive/evaluation/Final_balanced_personas_gpt_2_without_MMP" \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_gpt2_baseline_mean_n_variance_tier1_eval.csv"

/content/drive/MyDrive/vpl_llm_2
[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading Baseline mean_and_variance model from /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_mean_and_var_baseline_Film/all/mean_and_variance_gpt2__0_0.0001_cosine_2_0.0_seed0_last_checkpoint …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1479      59.2%       0.0995      0.6135
Persona B     2500      983      39.3%      

## Tier 2 - Simulated Persona Trasition Evaluation

In [8]:
!python /content/drive/MyDrive/vpl_llm_2/run_all_drift_evals.py \
    --model_type vae \
    --vae_model_path "/content/drive/MyDrive/vpl_llm_2/logs/gpt2_Persona_2_survey_100_film_MMP/all/vae_gpt2__0_0.0001_cosine_5_0.001_512_768_seed0_peft_last_checkpoint/model.pt" \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/FYP_final/grouped_personas_2_gpt_2_vpl_without_MMP_without_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt" \
    --build_sims \
    --source_sim_dir "/content/drive/MyDrive/evaluation/Final_balanced_personas_gpt2_MMP" \
    --sim_dir "/content/drive/MyDrive/evaluation/simulation/multi_drift_gpt2" \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_gpt_2_vpl_vs_ours.csv"


[Tier 3] Building simulation files …
Building 10 drift simulations in /content/drive/MyDrive/evaluation/simulation/multi_drift_gpt2 ...
  A->B: 45 rows written -> sim_A_to_B.jsonl  [seed=10, drift=20, stable=5, final=10]
  A->C: 45 rows written -> sim_A_to_C.jsonl  [seed=10, drift=20, stable=5, final=10]
  A->D: 45 rows written -> sim_A_to_D.jsonl  [seed=10, drift=20, stable=5, final=10]
  A->E: 45 rows written -> sim_A_to_E.jsonl  [seed=10, drift=20, stable=5, final=10]
  B->C: 45 rows written -> sim_B_to_C.jsonl  [seed=10, drift=20, stable=5, final=10]
  B->D: 45 rows written -> sim_B_to_D.jsonl  [seed=10, drift=20, stable=5, final=10]
  B->E: 45 rows written -> sim_B_to_E.jsonl  [seed=10, drift=20, stable=5, final=10]
  C->D: 45 rows written -> sim_C_to_D.jsonl  [seed=10, drift=20, stable=5, final=10]
  C->E: 45 rows written -> sim_C_to_E.jsonl  [seed=10, drift=20, stable=5, final=10]
  D->E: 45 rows written -> sim_D_to_E.jsonl  [seed=10, drift=20, stable=5, final=10]

Done.

TIER 3

In [9]:
!python run_all_drift_evals.py \
    --model_type categorical \
    --vae_model_path "/content/drive/MyDrive/vpl_llm_2/logs/gpt2_Persona_2_survey_100_film_MMP/all/vae_gpt2__0_0.0001_cosine_5_0.001_512_768_seed0_peft_last_checkpoint/model.pt" \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_categorical_baseline_Film/all/categorical_gpt2__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint" \
    --sim_dir "/content/drive/MyDrive/evaluation/simulation/multi_drift_gpt2" \
     --baseline_embed_dim 768  \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_gpt_2_categorical_vs_ours.csv"



TIER 3 — Z-Anchor Drift Tracking  (all persona-pair transitions)

[Tier 3] Loading models …
  VAE model: /content/drive/MyDrive/vpl_llm_2/logs/gpt2_Persona_2_survey_100_film_MMP/all/vae_gpt2__0_0.0001_cosine_5_0.001_512_768_seed0_peft_last_checkpoint/model.pt
  Baseline model (categorical): /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_categorical_baseline_Film/all/categorical_gpt2__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint
  Loading weights from: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_categorical_baseline_Film/all/categorical_gpt2__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint/pytorch_model.bin

── A → B  (sim_A_to_B.jsonl) ──
Model dtype: torch.bfloat16
[VAESession] Initialized with 8 pairs | window=8/8 | z_anchor_norm=1.4688
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=1.4688
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=1.4844
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=1.5078
[VA

In [10]:
!python run_all_drift_evals.py \
    --model_type base \
    --vae_model_path "/content/drive/MyDrive/vpl_llm_2/logs/gpt2_Persona_2_survey_100_film_MMP/all/vae_gpt2__0_0.0001_cosine_5_0.001_512_768_seed0_peft_last_checkpoint/model.pt" \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_base_baseline_Film/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint" \
 --sim_dir "/content/drive/MyDrive/evaluation/simulation/multi_drift_gpt2" \
      --baseline_embed_dim 768  \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_gpt_2_btl_vs_ours.csv"


TIER 3 — Z-Anchor Drift Tracking  (all persona-pair transitions)

[Tier 3] Loading models …
  VAE model: /content/drive/MyDrive/vpl_llm_2/logs/gpt2_Persona_2_survey_100_film_MMP/all/vae_gpt2__0_0.0001_cosine_5_0.001_512_768_seed0_peft_last_checkpoint/model.pt
  Baseline model (base): /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_base_baseline_Film/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint
  Loading weights from: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_base_baseline_Film/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint/pytorch_model.bin

── A → B  (sim_A_to_B.jsonl) ──
Model dtype: torch.bfloat16
[VAESession] Initialized with 8 pairs | window=8/8 | z_anchor_norm=1.4688
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=1.4688
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=1.4844
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=1.5078
[VAESession] z_anchor updated | window_size=8/8 | z_

In [11]:
!python run_all_drift_evals.py \
    --model_type mean_and_variance \
    --vae_model_path "//content/drive/MyDrive/vpl_llm_2/logs/gpt2_Persona_2_survey_100_film_MMP/all/vae_gpt2__0_0.0001_cosine_5_0.001_512_768_seed0_peft_last_checkpoint/model.pt" \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_mean_and_var_baseline_Film/all/mean_and_variance_gpt2__0_0.0001_cosine_2_0.0_seed0_last_checkpoint" \
 --sim_dir "/content/drive/MyDrive/evaluation/simulation/multi_drift_gpt2" \
      --baseline_embed_dim 768  \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_gpt_2_mean_var_vs_ours.csv"


TIER 3 — Z-Anchor Drift Tracking  (all persona-pair transitions)

[Tier 3] Loading models …
  VAE model: //content/drive/MyDrive/vpl_llm_2/logs/gpt2_Persona_2_survey_100_film_MMP/all/vae_gpt2__0_0.0001_cosine_5_0.001_512_768_seed0_peft_last_checkpoint/model.pt
  Baseline model (mean_and_variance): /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_mean_and_var_baseline_Film/all/mean_and_variance_gpt2__0_0.0001_cosine_2_0.0_seed0_last_checkpoint
  Loading weights from: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas__gpt_2_mean_and_var_baseline_Film/all/mean_and_variance_gpt2__0_0.0001_cosine_2_0.0_seed0_last_checkpoint/pytorch_model.bin

── A → B  (sim_A_to_B.jsonl) ──
Model dtype: torch.bfloat16
[VAESession] Initialized with 8 pairs | window=8/8 | z_anchor_norm=1.4688
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=1.4688
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=1.4844
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_

# Bechmarking with llama

## 2. Llama-3-8B Benchmarking

Scaling the VPL to a larger instruction-tuned model (Llama-3-8B-Instruct). This tests the architecture's portability and performance gains with higher-dimensional embeddings (4096).

Static VPL - logs/FYP_final/grouped_personas_5_witouth_MMP_without_Film_llama_VPL

### Analysis of Llama-3 Tier 1 Results

*   **Proposed Model:** Reached an impressive **75.3%** mean accuracy.
*   **Adaptation Velocity:** The Llama-3 results demonstrate that higher-capacity models allow for more distinct latent space separation, particularly for complex personas like D and E (both >84%).

In [12]:

!python /content/drive/MyDrive/vpl_llm_2/evaluate_reward_scores.py \
--vae_model_path  /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/grouped_personas_5_witouth_MMP_without_Film_llama_VPL/all/vae_llama-3-8b-instruct__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt \
--sim_dir         /content/drive/MyDrive/evaluation/Final_balanced_personas_llama3_without_MMP \
--n_context       8 \
--output_csv      /content/drive/MyDrive/evaluation/results/final_llama_without_MMP_without_film_eval.csv

[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading VAE model from /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/grouped_personas_5_witouth_MMP_without_Film_llama_VPL/all/vae_llama-3-8b-instruct__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1763      70.5%       0.2050      0.4039
Persona B     2500      992      39.7%      -0.0365      0.44

In [13]:
!python /content/drive/MyDrive/vpl_llm_2/evaluate_reward_scores.py \
--vae_model_path  /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt \
--sim_dir         /content/drive/MyDrive/evaluation/Final_balanced_Personas \
--n_context       8 \
--output_csv      /content/drive/MyDrive/evaluation/results/final_llama_with_Film_with_MMP_eval.csv

[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading VAE model from /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1822      72.9%       0.6990      1.3260
Persona B     2500     1777      71.1%       0.7036      1.3092
Persona C     2500     1596      63.8%   

In [14]:
%cd /content/drive/MyDrive/vpl_llm_2

!python evaluate_reward_scores.py \
    --model_type categorical \
    --baseline_embed_dim 4096 \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_categorical_llama/all/categorical_llama-3-8b-instruct__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint" \
    --sim_dir /content/drive/MyDrive/evaluation/Final_balanced_Personas \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_llama_baseline_categorical_tier1_eval.csv"

/content/drive/MyDrive/vpl_llm_2
[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading Baseline categorical model from /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_categorical_llama/all/categorical_llama-3-8b-instruct__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1424      57.0%       0.0207      0.0865
Persona B     2500     1485      59.4

In [15]:
%cd /content/drive/MyDrive/vpl_llm_2

!python evaluate_reward_scores.py \
    --model_type base \
    --baseline_embed_dim 4096 \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_base_llama/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint" \
    --sim_dir /content/drive/MyDrive/evaluation/Final_balanced_Personas \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_llama_baseline_BTL_tier1_eval..csv"

/content/drive/MyDrive/vpl_llm_2
[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading Baseline base model from /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_base_llama/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1445      57.8%       0.1005      0.4257
Persona B     2500     1461      58.4%       0.0945      0.4288
Persona C     25

In [16]:
%cd /content/drive/MyDrive/vpl_llm_2

!python evaluate_reward_scores.py \
    --model_type mean_and_variance \
    --baseline_embed_dim 4096 \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_llama/all/mean_and_variance_llama-3-8b-instruct__0_0.0001_cosine_2_0.0_seed0_last_checkpoint" \
    --sim_dir /content/drive/MyDrive/evaluation/Final_balanced_Personas \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_llama_baseline_mean_n_var_tier1_eval.csv"

/content/drive/MyDrive/vpl_llm_2
[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading Baseline mean_and_variance model from /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_llama/all/mean_and_variance_llama-3-8b-instruct__0_0.0001_cosine_2_0.0_seed0_last_checkpoint …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1446      57.8%       0.2107      1.1007
Persona B     2500     1486      59.4%  

## Tier 2 - Simulated Persona Trasition Evaluation

In [17]:
!python /content/drive/MyDrive/vpl_llm_2/run_all_drift_evals.py \
    --model_type vae \
    --vae_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt" \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/FYP_final/grouped_personas_5_witouth_MMP_without_Film_llama_VPL/all/vae_llama-3-8b-instruct__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt" \
    --build_sims \
    --source_sim_dir /content/drive/MyDrive/evaluation/Final_balanced_Personas \
    --sim_dir /content/drive/MyDrive/evaluation/simulation/multi_drift \
    --baseline_embed_dim 4096 \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_llama_vpl_vs_ours.csv"


[Tier 3] Building simulation files …
Building 10 drift simulations in /content/drive/MyDrive/evaluation/simulation/multi_drift ...
  A->B: 45 rows written -> sim_A_to_B.jsonl  [seed=10, drift=20, stable=5, final=10]
  A->C: 45 rows written -> sim_A_to_C.jsonl  [seed=10, drift=20, stable=5, final=10]
  A->D: 45 rows written -> sim_A_to_D.jsonl  [seed=10, drift=20, stable=5, final=10]
  A->E: 45 rows written -> sim_A_to_E.jsonl  [seed=10, drift=20, stable=5, final=10]
  B->C: 45 rows written -> sim_B_to_C.jsonl  [seed=10, drift=20, stable=5, final=10]
  B->D: 45 rows written -> sim_B_to_D.jsonl  [seed=10, drift=20, stable=5, final=10]
  B->E: 45 rows written -> sim_B_to_E.jsonl  [seed=10, drift=20, stable=5, final=10]
  C->D: 45 rows written -> sim_C_to_D.jsonl  [seed=10, drift=20, stable=5, final=10]
  C->E: 45 rows written -> sim_C_to_E.jsonl  [seed=10, drift=20, stable=5, final=10]
  D->E: 45 rows written -> sim_D_to_E.jsonl  [seed=10, drift=20, stable=5, final=10]

Done.

TIER 3 — Z-

In [18]:
!python run_all_drift_evals.py \
    --model_type categorical \
    --vae_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt" \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_categorical_llama/all/categorical_llama-3-8b-instruct__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint" \
    --sim_dir "/content/drive/MyDrive/evaluation/simulation/multi_drift" \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_llama_categorical_vs_ours.csv"



TIER 3 — Z-Anchor Drift Tracking  (all persona-pair transitions)

[Tier 3] Loading models …
  VAE model: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt
  Baseline model (categorical): /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_categorical_llama/all/categorical_llama-3-8b-instruct__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint
  Loading weights from: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_categorical_llama/all/categorical_llama-3-8b-instruct__0_0.0001_cosine_2_10_0.1_seed0_last_checkpoint/pytorch_model.bin

── A → B  (sim_A_to_B.jsonl) ──
Model dtype: torch.bfloat16
[VAESession] Initialized with 8 pairs | window=8/8 | z_anchor_norm=5.1875
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=5.2188
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=5.0625
[VAESession] z_anchor updated | window_

In [19]:
!python run_all_drift_evals.py \
    --model_type mean_and_variance \
    --vae_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt" \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_llama/all/mean_and_variance_llama-3-8b-instruct__0_0.0001_cosine_2_0.0_seed0_last_checkpoint" \
    --sim_dir "/content/drive/MyDrive/evaluation/simulation/multi_drift" \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_llama_mean_n_var_vs_ours.csv"



TIER 3 — Z-Anchor Drift Tracking  (all persona-pair transitions)

[Tier 3] Loading models …
  VAE model: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt
  Baseline model (mean_and_variance): /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_llama/all/mean_and_variance_llama-3-8b-instruct__0_0.0001_cosine_2_0.0_seed0_last_checkpoint
  Loading weights from: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_llama/all/mean_and_variance_llama-3-8b-instruct__0_0.0001_cosine_2_0.0_seed0_last_checkpoint/pytorch_model.bin

── A → B  (sim_A_to_B.jsonl) ──
Model dtype: torch.bfloat16
[VAESession] Initialized with 8 pairs | window=8/8 | z_anchor_norm=5.1875
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=5.2188
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=5.0625
[VAESession] z_anchor updated | window_size=8/8 | z

In [20]:
!python run_all_drift_evals.py \
    --model_type base \
    --vae_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt" \
    --baseline_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_base_llama/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint" \
    --sim_dir "/content/drive/MyDrive/evaluation/simulation/multi_drift" \
    --output_csv "/content/drive/MyDrive/evaluation/results/final_llama_btl_vs_ours.csv"



TIER 3 — Z-Anchor Drift Tracking  (all persona-pair transitions)

[Tier 3] Loading models …
  VAE model: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt
  Baseline model (base): /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_base_llama/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint
  Loading weights from: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_All_baeline_Film_base_llama/all/base_gpt2__0_0.0001_cosine_2_seed0_last_checkpoint/pytorch_model.bin

── A → B  (sim_A_to_B.jsonl) ──
Model dtype: torch.bfloat16
[VAESession] Initialized with 8 pairs | window=8/8 | z_anchor_norm=5.1875
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=5.2188
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=5.0625
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=4.8750
[VAESession] z_anchor updated | window_size=8/8

# Evaluate Adaption

## 3. Tier 3: Drift and Adaptation Analysis

Evaluating the **Z-Anchor** mechanism's ability to track persona shifts in real-time. We observe the `z_anchor_norm` as it adapts to transitions (e.g., A → B).

In [21]:
!python evaluate_adaptation_velocity.py \
    --vae_model_path "/content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt" \
    --sim_dir  "/content/drive/MyDrive/evaluation/simulation/multi_drift" \
    --vae_dev "cuda:0" \
    --output_dir "/content/drive/MyDrive/evaluation/results/adaptation"



[Velocity] Loading VAE model: /content/drive/MyDrive/vpl_llm_2/logs/grouped_personas_100_film_final_01/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_4096_seed0_peft_last_checkpoint/model.pt
[Velocity] Loading 10 simulation files...
[Velocity] Loading 10 simulation files...

[Velocity] Running sweeps over momentums …
  Momentum 0.0 …
Model dtype: torch.bfloat16
[VAESession] Initialized with 8 pairs | window=8/8 | z_anchor_norm=5.1875
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=5.3750
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=4.3125
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=4.2812
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=4.7500
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=4.7188
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=5.8438
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=5.7188
[VAESession] z_anchor updated | window_size=8/8 | z_anchor_norm=6.593

## 4. Ablation Study

Systematic removal of components to quantify their contribution:
1.  **-MMP + FiLm:** Tests if FiLm alone handles conditioning.
2.  **+MMP - FiLm:** Tests the raw latent space without modulated activations.
3.  **-MMP - FiLm:** Baseline VPL.
4.  **+MMP + FiLm:** Ours

In [22]:
!python /content/drive/MyDrive/vpl_llm_2/evaluate_reward_scores.py \
--vae_model_path  /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/ablation_study_grouped_personas_2_gpt_2_vpl_with_MMP_with_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt \
--sim_dir         /content/drive/MyDrive/evaluation/Final_balanced_personas_gpt2_MMP \
--n_context       8 \
--output_csv      /content/drive/MyDrive/evaluation/results/final_gpt2_without_MMP_with_FiLm_eval.csv

[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading VAE model from /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/ablation_study_grouped_personas_2_gpt_2_vpl_with_MMP_with_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1565      62.6%       0.2512      0.7613
Persona B     2500     1489      59.6%       0.2856      0.9640
Pers

In [23]:
!python /content/drive/MyDrive/vpl_llm_2/evaluate_reward_scores.py \
--vae_model_path /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/ablation_study_grouped_personas_2_gpt_2_vpl_with_MMP_without_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt\
--sim_dir         /content/drive/MyDrive/evaluation/Final_balanced_personas_gpt_2_without_MMP \
--n_context       8 \
--output_csv      /content/drive/MyDrive/evaluation/results/final_gpt2_with_MMP_without_FiLm_eval.csv

[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading VAE model from /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/ablation_study_grouped_personas_2_gpt_2_vpl_with_MMP_without_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1492      59.7%       0.1794      0.7096
Persona B     2500     1179      47.2%      -0.0726      0.7106
P

In [24]:
!python /content/drive/MyDrive/vpl_llm_2/evaluate_reward_scores.py \
--vae_model_path  /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/ablation_study_grouped_personas_2_gpt_2_vpl_without_MMP_without_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt \
--sim_dir          /content/drive/MyDrive/evaluation/Final_balanced_personas_gpt_2_without_MMP \
--n_context       8 \
--output_csv      /content/drive/MyDrive/evaluation/results/final_gpt2_without_MMP_without_FiLm_eval.csv

[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading VAE model from /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/ablation_study_grouped_personas_2_gpt_2_vpl_without_MMP_without_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1697      67.9%       0.1254      0.3117
Persona B     2500      909      36.4%      -0.0506      0.325

In [25]:
!python /content/drive/MyDrive/vpl_llm_2/evaluate_reward_scores.py \
--vae_model_path /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/ablation_study_grouped_personas_2_gpt_2_vpl_without_MMP_with_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt \
--sim_dir         /content/drive/MyDrive/evaluation/Final_balanced_personas_gpt_2_without_MMP \
--n_context       8 \
--output_csv      /content/drive/MyDrive/evaluation/results/final_gpt2_without_MMP_with_FiLm_eval.csv

[Setup] Loading persona test files …
  Persona A: 2500 rows loaded from test_Persona_A.jsonl
  Persona B: 2500 rows loaded from test_Persona_B.jsonl
  Persona C: 2500 rows loaded from test_Persona_C.jsonl
  Persona D: 2500 rows loaded from test_Persona_D.jsonl
  Persona E: 2500 rows loaded from test_Persona_E.jsonl

[Setup] Loading VAE model from /content/drive/MyDrive/vpl_llm_2/logs/FYP_final/ablation_study_grouped_personas_2_gpt_2_vpl_without_MMP_with_Film/all/vae_gpt2__0_0.0001_cosine_2_0.001_512_768_seed0_peft_last_checkpoint/model.pt …
  Persona A: 2500 test pairs
  Persona B: 2500 test pairs
  Persona C: 2500 test pairs
  Persona D: 2500 test pairs
  Persona E: 2500 test pairs

TIER 1 — Per-Persona Reward Accuracy  (rc > rr ?)
Persona        N  Correct   Accuracy  Mean(rc-rr)  Std(rc-rr)
-----------------------------------------------------------------
Persona A     2500     1385      55.4%       0.0637      0.4659
Persona B     2500     1380      55.2%       0.0294      0.2389
P